In [1]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00


In [11]:
# 1. Install dependencies
# !pip install -q transformers peft trl datasets bitsandbytes accelerate

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# --- Configuration ---
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATA_PATH = "train.jsonl"
OUTPUT_DIR = "adapters/"

print("1. Loading Day 1 Dataset...")
dataset = load_dataset("json", data_files={"train": DATA_PATH}, split="train")

def format_prompt(example):
    """Formats the instruction, input, and output into a single training string."""
    text = f"User: {example['instruction']}\nContext: {example['input']}\nAssistant: {example['output']}"
    return {"text": text}

dataset = dataset.map(format_prompt)

print("2. Bypassing Quantization (Model fits natively on T4!)...")
# We removed BitsAndBytes! TinyLlama is 1.1B params (~2.2GB in float16).
# A Colab T4 has 15GB of VRAM, so it fits perfectly without 4-bit compression.

print("3. Loading Model and Tokenizer natively in Float16...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16 # Standard 16-bit native load
)

# Enable memory-saving gradient checkpointing
model.gradient_checkpointing_enable()

print("4. Setting up LoRA (PEFT)...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Wrap the base model with LoRA adapters
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("5. Configuring Training Arguments...")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=3,
    optim="paged_adamw_8bit",
    fp16=True,
    save_strategy="epoch",
    max_length=512,
    dataset_text_field="text"
)

print("6. Initializing SFTTrainer and Starting Training...")
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)

# Start the actual training!
trainer.train()

print(f"\n7. Saving final adapter weights to {OUTPUT_DIR}...")
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Day 2 Training Complete!")

1. Loading Day 1 Dataset...


Map:   0%|          | 0/1744 [00:00<?, ? examples/s]

2. Bypassing Quantization (Model fits natively on T4!)...
3. Loading Model and Tokenizer natively in Float16...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

4. Setting up LoRA (PEFT)...
trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044
5. Configuring Training Arguments...
6. Initializing SFTTrainer and Starting Training...


...

Tokenizing train dataset:   0%|          | 0/1744 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1744 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.146053
20,0.885310
30,0.781106
40,0.700224
50,0.739292
60,0.727342
70,0.720775
80,0.741032
90,0.675935
100,0.697151



7. Saving final adapter weights to adapters/...
✅ Day 2 Training Complete!
